[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/06-dsa-interview/10_graphs.ipynb)

*DSA Interview Handbook · Notebook 10 of 14*

# Graphs

BFS, DFS, topological sort, union-find and shortest paths — on grids and on adjacency lists.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A graph is a map of cities and roads. BFS spreads like a ripple; DFS follows one road to its end.

**What interviewers really test here:** Modelling a problem as a graph, cycle detection, topological sort and Dijkstra.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. What a graph is and how to store it *(Basics)*
- [ ] 2. Depth-first search *(Basics)*
- [ ] 3. Breadth-first search and shortest paths *(Core)*
- [ ] 4. Grids: islands and flood fill *(Core)*
- [ ] 5. Directed graphs: topological order and cycles *(Core)*
- [ ] 6. Union–Find (disjoint set union) *(Core)*
- [ ] 7. Weighted shortest paths *(Advanced)*
- [ ] 8. Spanning trees and two-colouring *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definition

A **graph** G = (V, E) is a non-linear data structure consisting of a set of **vertices** (nodes) V and a set of **edges** E, where each edge connects a pair of vertices. Graphs model relationships and networks.



A tree is a special graph: connected, undirected and acyclic, with exactly V − 1 edges.

### Terminology

| Term | Meaning |
| --- | --- |
| Adjacent | two vertices joined by an edge |
| Degree | number of edges at a vertex; in directed graphs, **in-degree** and **out-degree** |
| Path | sequence of vertices joined by edges; **simple** if no vertex repeats |
| Cycle | a path that starts and ends at the same vertex |
| Connected graph | a path exists between every pair of vertices (undirected) |
| Connected component | a maximal connected piece of an undirected graph |
| Strongly connected | directed graph where every vertex reaches every other |
| DAG | directed acyclic graph |
| Spanning tree | a tree using all vertices and a subset of the edges |
| Bipartite | vertices split into two sets with every edge crossing between them |



**Handshake lemma**: in an undirected graph, the sum of all degrees = 2|E| (each edge counts at both ends).

### Types of graphs

| Type | Description | Example |
| --- | --- | --- |
| Undirected | edges have no direction | friendship, roads both ways |
| Directed (digraph) | edges point from u to v | followers, prerequisites, web links |
| Weighted | edges carry a cost | distances, latency, prices |
| Unweighted | all edges equal | grid moves, word ladder |
| Cyclic / acyclic | has / has no cycle | DAG of tasks |
| Sparse / dense | E ≈ V vs E ≈ V² | road networks vs complete graphs |
| Complete | every pair connected: V(V − 1)/2 edges | — |
| Implicit | edges computed on the fly, not stored | grids, puzzle states |

### Representations

|   | Adjacency matrix | Adjacency list | Edge list |
| --- | --- | --- | --- |
| Space | O(V²) | O(V + E) | O(E) |
| Is (u, v) an edge? | O(1) | O(deg u) | O(E) |
| List neighbours of u | O(V) | O(deg u) | O(E) |
| Add edge | O(1) | O(1) | O(1) |
| Best for | dense graphs, Floyd–Warshall | most algorithms (BFS, DFS, Dijkstra) | Kruskal, Bellman–Ford, input format |

### Traversals: BFS and DFS

|   | Breadth-first search | Depth-first search |
| --- | --- | --- |
| Data structure | queue | stack / recursion |
| Order | level by level (by distance) | as deep as possible, then backtrack |
| Shortest path (unweighted) | **yes** | no |
| Time | O(V + E) | O(V + E) |
| Space | O(V) — the frontier can be wide | O(V) — the recursion can be deep |
| Uses | shortest steps, levels, multi-source spread | components, cycle detection, topological sort, path finding, backtracking |



**Cycle detection**: undirected — a visited neighbour that isn't the parent means a cycle (or union-find). Directed — reaching a vertex that is still on the current DFS path (colour "grey") means a cycle.

### Core algorithms

| Algorithm | Solves | Requirement | Time |
| --- | --- | --- | --- |
| BFS | single-source shortest path | unweighted | O(V + E) |
| 0-1 BFS | shortest path | weights 0 or 1 (deque) | O(V + E) |
| **Dijkstra** | single-source shortest path | non-negative weights | O((V + E) log V) |
| **Bellman–Ford** | single-source shortest path, detects negative cycles | any weights | O(V · E) |
| Floyd–Warshall | all-pairs shortest paths | no negative cycles | O(V³) |
| **Topological sort** (Kahn / DFS) | ordering of a DAG | directed, acyclic | O(V + E) |
| **Kruskal** | minimum spanning tree | undirected, weighted | O(E log E) |
| **Prim** | minimum spanning tree | undirected, weighted | O(E log V) heap, O(V²) array |
| Union–Find | dynamic connectivity | — | ≈ O(α(n)) per operation |
| Kosaraju / Tarjan | strongly connected components | directed | O(V + E) |



**Why Dijkstra fails with negative edges**: it finalises the closest vertex assuming no later path can be shorter; a negative edge discovered later could make it shorter.

### Applications

- Maps and navigation (shortest routes: Dijkstra, A\*).
- Social networks (friend suggestions, communities, degrees of separation).
- Computer networks (routing protocols use Bellman–Ford / Dijkstra; spanning tree protocol).
- Build systems, package managers and course planning (topological sort of dependencies).
- Web crawling and PageRank (directed link graph).
- Circuit and pipeline design (minimum spanning trees).
- Recommendation systems and knowledge graphs.
- Image processing (flood fill, connected regions in a pixel grid).

---
# Part 2: Lessons, step by step

## Lesson 1: What a graph is and how to store it

*Level: Basics*

A **graph** is a set of **nodes** (vertices) joined by **edges**. Cities and roads, people and friendships, courses and prerequisites, web pages and links — all graphs.




- **Undirected** edges go both ways (friendship). **Directed** edges have an arrow (prerequisite → course).
- **Weighted** edges carry a cost (distance, time).
- A **path** is a sequence of edges; a **cycle** is a path that returns to its start.



#### Three ways to store one



| Form | Looks like | Good for |
| --- | --- | --- |
| Edge list | `[(0,1), (1,2), (0,2)]` | input format, Kruskal, union-find |
| **Adjacency list** | `{0: [1, 2], 1: [0, 2], …}` | almost everything — O(V + E) space |
| Adjacency matrix | `m[u][v] = 1` | dense graphs, O(1) "is there an edge?" |



A **grid** is a graph too: each cell is a node connected to its 4 neighbours. You don't build anything — just compute neighbours with `(r ± 1, c)`, `(r, c ± 1)`.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def adjacency_lesson():
    edges = [(0, 1), (0, 2), (1, 3), (2, 3), (3, 4)]
    adj = {i: [] for i in range(5)}
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)
    return adj


adjacency_lesson()

{0: [1, 2], 1: [0, 3], 2: [0, 3], 3: [1, 2, 4], 4: [3]}

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Edge 0–1: add 1 to 0's list and 0 to 1's list
2. Edge 0–2: add 2 to 0's list and 0 to 2's list
3. Edge 1–3: add 3 to 1's list and 1 to 3's list
4. Edge 2–3: add 3 to 2's list and 2 to 3's list
5. Edge 3–4: add 4 to 3's list and 3 to 4's list

</details>


## Lesson 2: Depth-first search

*Level: Basics*

**DFS** goes as deep as possible along one path before backing up — like exploring a maze by always taking the next unexplored turn. Mark nodes as **visited** so you never loop forever.



```python
seen = set()
def dfs(u):
    seen.add(u)
    for v in adj[u]:
        if v not in seen:
            dfs(v)
```



Iterative version: replace recursion with a stack (`stack.pop()` / `stack.append()`). Each node and edge is handled once → **O(V + E)**.



DFS answers "can I reach…?", "how many connected pieces?", "is there a cycle?", and is the base of topological sort.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def dfs_lesson():
    edges = [(0, 1), (0, 2), (1, 3), (2, 4), (3, 4), (4, 5)]
    adj = {i: [] for i in range(6)}
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)
    seen, order = set(), []

    def dfs(u):
        seen.add(u)
        order.append(u)
        for v in adj[u]:
            if v not in seen:
                dfs(v)

    dfs(0)
    return order


dfs_lesson()

[0, 1, 3, 4, 2, 5]

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Visit 0; go deeper into an unvisited neighbour
2. Visit 1; go deeper into an unvisited neighbour
3. Visit 3; go deeper into an unvisited neighbour
4. Visit 4; go deeper into an unvisited neighbour
5. Visit 2; go deeper into an unvisited neighbour
6. Visit 5; go deeper into an unvisited neighbour

</details>


**✏️ Your turn:** you now have what you need for [Q8](#q8), [Q1](#q1). Try one now, or carry on and do them in Part 3.

## Lesson 3: Breadth-first search and shortest paths

*Level: Core*

**BFS** explores in rings: first all nodes 1 edge away, then 2 edges, then 3. It uses a queue. Because of that order, the first time BFS reaches a node is along a **shortest path** (when every edge costs the same).



```python
dist = {start: 0}
q = deque([start])
while q:
    u = q.popleft()
    for v in adj[u]:
        if v not in dist:          # mark when you PUSH
            dist[v] = dist[u] + 1
            q.append(v)
```



**Multi-source BFS**: put every starting point in the queue at once (all rotten oranges, all zeros). Each layer then means "one more minute" or "one more step from the nearest source".

**✏️ Your turn:** you now have what you need for [Q4](#q4), [Q5](#q5), [Q15](#q15), [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 4: Grids: islands and flood fill

*Level: Core*

Most grid problems are "count or measure the connected groups". Loop over every cell; when you find an unvisited land cell, that's a new group — run DFS/BFS from it and mark everything it reaches.



```python
for r in range(m):
    for c in range(n):
        if grid[r][c] == "1":
            count += 1
            sink(r, c)          # DFS that turns the whole island to "0"
```



Useful twists:




- **Return a size** from the DFS for max area.
- **Search from the border** when the question is about regions *not* touching the edge (surrounded regions), or reaching an ocean (Pacific–Atlantic: climb uphill from each ocean).

**✏️ Your turn:** you now have what you need for [Q2](#q2), [Q3](#q3), [Q6](#q6), [Q7](#q7), [Q1](#q1). Try one now, or carry on and do them in Part 3.

## Lesson 5: Directed graphs: topological order and cycles

*Level: Core*

A **topological order** lists the nodes of a directed graph so every edge points forward — "take prerequisites before courses". It exists only if there's **no cycle**.



#### Kahn's algorithm




1. Count each node's **in-degree** (incoming edges).
2. Queue every node with in-degree 0 — nothing blocks it.
3. Pop a node, add it to the order, and decrease the in-degree of its neighbours; push any that reach 0.
4. If the order has all V nodes, done. If not, the leftovers sit on a cycle.



DFS alternative: colour nodes 0 = new, 1 = on the current path, 2 = finished. Reaching a node coloured 1 means a cycle (used in "eventual safe states").

**✏️ Your turn:** you now have what you need for [Q10](#q10), [Q11](#q11), [Q12](#q12). Try one now, or carry on and do them in Part 3.

## Lesson 6: Union–Find (disjoint set union)

*Level: Core*

**Union–Find** keeps track of groups as you connect things. Each node points to a parent; the node at the top of a chain is the group's **root**.



```python
parent = list(range(n))
def find(x):                       # root of x's group
    while parent[x] != x:
        parent[x] = parent[parent[x]]   # path compression
        x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra == rb: return False       # already connected → this edge makes a cycle
    parent[ra] = rb
    return True
```



With path compression (and optionally union by size), each operation is nearly O(1). Use it for counting groups, detecting the edge that closes a cycle, merging accounts, and Kruskal's MST.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def union_find_lesson():
    parent = list(range(6))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    edges = [(0, 1), (2, 3), (1, 3), (4, 5), (0, 2)]
    done = []
    for a, b in edges:
        ra, rb = find(a), find(b)
        done.append((a, b))
        if ra == rb:
            pass
        else:
            parent[ra] = rb
    return parent


union_find_lesson()

[3, 3, 3, 3, 5, 5]

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. union(0, 1): root 0 now points to root 1
2. union(2, 3): root 2 now points to root 3
3. union(1, 3): root 1 now points to root 3
4. union(4, 5): root 4 now points to root 5
5. 0 and 2 already share root 3 → this edge would close a cycle

</details>


**✏️ Your turn:** you now have what you need for [Q8](#q8), [Q13](#q13), [Q14](#q14). Try one now, or carry on and do them in Part 3.

## Lesson 7: Weighted shortest paths

*Level: Advanced*

When edges have different costs, BFS is no longer enough.



#### Dijkstra (no negative weights)



Keep a min-heap of `(distance so far, node)`. Pop the closest node — its distance is now final — and relax its edges. Skip stale heap entries. O(E log V).



```python
dist = {}
heap = [(0, src)]
while heap:
    d, u = heapq.heappop(heap)
    if u in dist: continue          # already finalised
    dist[u] = d
    for v, w in adj[u]:
        if v not in dist:
            heapq.heappush(heap, (d + w, v))
```



#### Bellman–Ford



Relax every edge, V − 1 rounds. Slower (O(V · E)) but handles negative weights, and running only k + 1 rounds gives "cheapest path with at most k stops" — copy the distance array each round.



The "cost" can be anything that only grows along a path: the max height difference (path with minimum effort), probability, time.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [5]:
def dijkstra_lesson():
    import heapq
    edges = [(0, 1, 4), (0, 2, 1), (2, 1, 2), (1, 3, 1), (2, 3, 5)]
    adj = {i: [] for i in range(4)}
    for u, v, w in edges:
        adj[u].append((v, w))
    dist, heap = {}, [(0, 0)]
    while heap:
        d, u = heapq.heappop(heap)
        if u in dist:
            continue
        dist[u] = d
        for v, w in adj[u]:
            if v not in dist:
                heapq.heappush(heap, (d + w, v))
    return dist


dijkstra_lesson()

{0: 0, 2: 1, 1: 3, 3: 4}

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Closest unfinished node is 0 at distance 0 → final
2. Closest unfinished node is 2 at distance 1 → final
3. Closest unfinished node is 1 at distance 3 → final
4. Pop (4, 1) but 1 is already final → skip stale entry
5. Closest unfinished node is 3 at distance 4 → final
6. Pop (6, 3) but 3 is already final → skip stale entry

</details>


**✏️ Your turn:** you now have what you need for [Q16](#q16), [Q17](#q17), [Q18](#q18). Try one now, or carry on and do them in Part 3.

## Lesson 8: Spanning trees and two-colouring

*Level: Advanced*

#### Minimum spanning tree



Connect all nodes with the cheapest total edge cost (no cycles needed).




- **Kruskal**: sort edges by cost; add each edge whose ends are in different union-find groups. O(E log E).
- **Prim**: grow one tree; repeatedly add the cheapest edge leaving it. With a plain array it's O(V²) — best for dense graphs like "connect all points".



#### Bipartite check



Can nodes be split into two teams with every edge going across? BFS and colour neighbours with the opposite colour; a same-colour edge means no. A graph is bipartite exactly when it has no odd-length cycle.

**✏️ Your turn:** you now have what you need for [Q19](#q19), [Q9](#q9). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Flood Fill

🟢 Warm-up · *DFS from the start pixel*

Recolour the region of same-coloured pixels connected to `(sr, sc)` with `color`.

**Examples**

- `image = [[1, 1, 1], [1, 1, 0], [1, 0, 1]], sr = 1, sc = 1, color = 2` → `[[2, 2, 2], [2, 2, 0], [2, 0, 1]]`
- `image = [[0, 0, 0], [0, 0, 0]], sr = 0, sc = 0, color = 0` → `[[0, 0, 0], [0, 0, 0]]`

In [ ]:
def flood_fill(image, sr, sc, color):
    """
    Recolour the region of same-coloured pixels connected to (sr, sc) with
    color.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(flood_fill, [
    (([[1, 1, 1], [1, 1, 0], [1, 0, 1]], 1, 1, 2), [[2, 2, 2], [2, 2, 0], [2, 0, 1]]),
    (([[0, 0, 0], [0, 0, 0]], 0, 0, 0), [[0, 0, 0], [0, 0, 0]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recolour the start, then keep sweeping the whole image recolouring any original-coloured pixel next to a recoloured one, until nothing changes.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

DFS from (sr, sc) recolouring pixels of the original colour. If the new colour equals the old one, return immediately.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Recolour the start, then keep sweeping the whole image recolouring any original-coloured pixel next to a recoloured one, until nothing changes.

```python
def flood_fill(image, sr, sc, color):
    old = image[sr][sc]
    if old == color:
        return image
    m, n = len(image), len(image[0])
    done = {(sr, sc)}
    changed = True
    while changed:
        changed = False
        for r in range(m):
            for c in range(n):
                if (r, c) not in done and image[r][c] == old and any((r + dr, c + dc) in done for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1))):
                    done.add((r, c))
                    changed = True
    for r, c in done:
        image[r][c] = color
    return image
```

**Cost:** O((m·n)²) time, O(m·n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** DFS from (sr, sc) recolouring pixels of the original colour. If the new colour equals the old one, return immediately.

```python
def flood_fill(image, sr, sc, color):
    old = image[sr][sc]
    if old == color:
        return image
    m, n = len(image), len(image[0])

    def dfs(r, c):
        if not (0 <= r < m and 0 <= c < n) or image[r][c] != old:
            return
        image[r][c] = color
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            dfs(r + dr, c + dc)

    dfs(sr, sc)
    return image
```

**Cost:** O(m·n) time, O(m·n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recolour the start, then keep sweeping the whole image recolouring any original-coloured pixel next to a recoloured one, until nothing changes. That costs O((m·n)²) time and O(m·n) space."

"I can do better: DFS from (sr, sc) recolouring pixels of the original colour. If the new colour equals the old one, return immediately. That's O(m·n) time and O(m·n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `image = [[1, 1, 1], [1, 1, 0], [1, 0, 1]], sr = 1, sc = 1, color = 2`

1. Paint (1,1)
2. Paint (0,1)
3. Paint (0,2)
4. Paint (0,0)
5. Paint (1,0)
6. Paint (2,0)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/graphs/flood-fill)

<a id="q2"></a>
### Q2. Number of Islands

🟡 Interview standard · *DFS flood fill*

Count the islands ('1' cells connected up/down/left/right) in a grid of '1' and '0'.

**Examples**

- `grid = [['1', '1', '1', '1', '0'], ['1', '1', '0', '1', '0'], ['1', '1', '0', '0', '0'], ['0', '0', '0', '0', '0']]` → `1`
- `grid = [['1', '1', '0', '0', '0'], ['1', '1', '0', '0', '0'], ['0', '0', '1', '0', '0'], ['0', '0', '0', '1', '1']]` → `3`

In [ ]:
def num_islands(grid):
    """
    Count the islands ('1' cells connected up/down/left/right) in a grid of '1'
    and '0'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(num_islands, [
    (([['1', '1', '1', '1', '0'], ['1', '1', '0', '1', '0'], ['1', '1', '0', '0', '0'], ['0', '0', '0', '0', '0']],), 1),
    (([['1', '1', '0', '0', '0'], ['1', '1', '0', '0', '0'], ['0', '0', '1', '0', '0'], ['0', '0', '0', '1', '1']],), 3),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each land cell, BFS from scratch to see whether it reaches any earlier land cell; if not, it starts a new island.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Scan the grid; each unvisited '1' starts a new island. Flood-fill it (sink it to '0') so it is never counted again.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For each land cell, BFS from scratch to see whether it reaches any earlier land cell; if not, it starts a new island.

```python
def num_islands(grid):
    from collections import deque
    m, n = len(grid), len(grid[0])
    land = [(r, c) for r in range(m) for c in range(n) if grid[r][c] == "1"]
    count = 0
    for k, (r, c) in enumerate(land):
        earlier = set(land[:k])
        seen, q, joined = {(r, c)}, deque([(r, c)]), False
        while q and not joined:
            x, y = q.popleft()
            for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nx, ny = x + dx, y + dy
                if 0 <= nx < m and 0 <= ny < n and grid[nx][ny] == "1" and (nx, ny) not in seen:
                    if (nx, ny) in earlier:
                        joined = True
                    seen.add((nx, ny))
                    q.append((nx, ny))
        count += not joined
    return count
```

**Cost:** O((m·n)²) time, O(m·n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Scan the grid; each unvisited '1' starts a new island. Flood-fill it (sink it to '0') so it is never counted again.

```python
def num_islands(grid):
    m, n = len(grid), len(grid[0])
    sunk = []

    def sink(r, c):
        if not (0 <= r < m and 0 <= c < n) or grid[r][c] != "1":
            return
        grid[r][c] = "0"
        sunk.append((r, c))
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            sink(r + dr, c + dc)

    count = 0
    for r in range(m):
        for c in range(n):
            if grid[r][c] == "1":
                count += 1
                start = len(sunk)
                sink(r, c)
    return count
```

**Cost:** O(m·n) time, O(m·n) recursion space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each land cell, BFS from scratch to see whether it reaches any earlier land cell; if not, it starts a new island. That costs O((m·n)²) time and O(m·n) space."

"I can do better: Scan the grid; each unvisited '1' starts a new island. Flood-fill it (sink it to '0') so it is never counted again. That's O(m·n) time and O(m·n) recursion space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `grid = [['1', '1', '0', '0', '1'], ['1', '0', '0', '1', '1'], ['0', '0', '1', '0', '0'], ['1', '0', '0', '0', '1']]`

1. New island #1 at (0,0) — flood it so it's not counted again
2. New island #2 at (0,4) — flood it so it's not counted again
3. New island #3 at (2,2) — flood it so it's not counted again
4. New island #4 at (3,0) — flood it so it's not counted again
5. New island #5 at (3,4) — flood it so it's not counted again

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/graphs/number-of-islands)

<a id="q3"></a>
### Q3. Max Area of Island

🟡 Interview standard · *DFS returning area*

Return the area of the largest island (connected 1s) in a binary grid, or 0.

**Examples**

- `grid = [[0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0], [0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0], [0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0]]` → `6`
- `grid = [[0, 0, 0, 0, 0, 0, 0, 0]]` → `0`

In [ ]:
def max_area_of_island(grid):
    """
    Return the area of the largest island (connected 1s) in a binary grid, or 0.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(max_area_of_island, [
    (([[0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0], [0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0], [0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0]],), 6),
    (([[0, 0, 0, 0, 0, 0, 0, 0]],), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

From every land cell, BFS its whole island from scratch to measure it.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

DFS from each unvisited land cell, sinking cells and returning 1 + the areas of the neighbours.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** From every land cell, BFS its whole island from scratch to measure it.

```python
def max_area_of_island(grid):
    from collections import deque
    m, n, best = len(grid), len(grid[0]), 0
    for r in range(m):
        for c in range(n):
            if grid[r][c]:
                seen, q = {(r, c)}, deque([(r, c)])
                while q:
                    x, y = q.popleft()
                    for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                        nx, ny = x + dx, y + dy
                        if 0 <= nx < m and 0 <= ny < n and grid[nx][ny] and (nx, ny) not in seen:
                            seen.add((nx, ny))
                            q.append((nx, ny))
                best = max(best, len(seen))
    return best
```

**Cost:** O((m·n)²) time, O(m·n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** DFS from each unvisited land cell, sinking cells and returning 1 + the areas of the neighbours.

```python
def max_area_of_island(grid):
    m, n = len(grid), len(grid[0])

    def area(r, c):
        if not (0 <= r < m and 0 <= c < n) or grid[r][c] != 1:
            return 0
        grid[r][c] = 2
        return 1 + sum(area(r + dr, c + dc) for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)))

    best = 0
    for r in range(m):
        for c in range(n):
            if grid[r][c] == 1:
                a = area(r, c)
                best = max(best, a)
    return best
```

**Cost:** O(m·n) time, O(m·n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: From every land cell, BFS its whole island from scratch to measure it. That costs O((m·n)²) time and O(m·n) space."

"I can do better: DFS from each unvisited land cell, sinking cells and returning 1 + the areas of the neighbours. That's O(m·n) time and O(m·n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/graphs/max-area-of-island)

<a id="q4"></a>
### Q4. Rotting Oranges

🟡 Interview standard · *Multi-source BFS*

Each minute, rotten oranges (2) rot their fresh neighbours (1). Return the minutes until none are fresh, or −1.

**Examples**

- `grid = [[2, 1, 1], [1, 1, 0], [0, 1, 1]]` → `4`
- `grid = [[2, 1, 1], [0, 1, 1], [1, 0, 1]]` → `-1`
- `grid = [[0, 2]]` → `0`

In [ ]:
def oranges_rotting(grid):
    """
    Each minute, rotten oranges (2) rot their fresh neighbours (1). Return the
    minutes until none are fresh, or −1.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(oranges_rotting, [
    (([[2, 1, 1], [1, 1, 0], [0, 1, 1]],), 4),
    (([[2, 1, 1], [0, 1, 1], [1, 0, 1]],), -1),
    (([[0, 2]],), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Simulate minute by minute, scanning the whole grid each minute.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Put every rotten orange in the queue at once and BFS layer by layer; each layer is one minute. Count fresh oranges to detect −1.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Simulate minute by minute, scanning the whole grid each minute.

```python
def oranges_rotting(grid):
    m, n, minutes = len(grid), len(grid[0]), 0
    while True:
        new = [(r, c) for r in range(m) for c in range(n) if grid[r][c] == 1 and any(0 <= r + dr < m and 0 <= c + dc < n and grid[r + dr][c + dc] == 2 for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)))]
        if not new:
            break
        for r, c in new:
            grid[r][c] = 2
        minutes += 1
    return -1 if any(1 in row for row in grid) else minutes
```

**Cost:** O((m·n)²) time, O(m·n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Put every rotten orange in the queue at once and BFS layer by layer; each layer is one minute. Count fresh oranges to detect −1.

```python
def oranges_rotting(grid):
    from collections import deque
    m, n = len(grid), len(grid[0])
    q = deque((r, c) for r in range(m) for c in range(n) if grid[r][c] == 2)
    fresh = sum(row.count(1) for row in grid)
    minutes = 0
    while q and fresh:
        layer = []
        for _ in range(len(q)):
            r, c = q.popleft()
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] == 1:
                    grid[nr][nc] = 2
                    fresh -= 1
                    q.append((nr, nc))
                    layer.append((nr, nc))
        minutes += 1
    return -1 if fresh else minutes
```

**Cost:** O(m·n) time, O(m·n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Simulate minute by minute, scanning the whole grid each minute. That costs O((m·n)²) time and O(m·n) space."

"I can do better: Put every rotten orange in the queue at once and BFS layer by layer; each layer is one minute. Count fresh oranges to detect −1. That's O(m·n) time and O(m·n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `grid = [[2, 1, 1], [1, 1, 0], [0, 1, 1]]`

1. Minute 0: all rotten oranges start in the queue
2. Minute 1: 2 orange(s) rot
3. Minute 2: 2 orange(s) rot
4. Minute 3: 1 orange(s) rot
5. Minute 4: 1 orange(s) rot

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/graphs/rotting-oranges)

<a id="q5"></a>
### Q5. 01 Matrix

🟡 Interview standard · *Multi-source BFS from zeros*

Return a matrix of each cell's distance to the nearest 0.

**Examples**

- `mat = [[0, 0, 0], [0, 1, 0], [0, 0, 0]]` → `[[0, 0, 0], [0, 1, 0], [0, 0, 0]]`
- `mat = [[0, 0, 0], [0, 1, 0], [1, 1, 1]]` → `[[0, 0, 0], [0, 1, 0], [1, 2, 1]]`

In [ ]:
def update_matrix(mat):
    """
    Return a matrix of each cell's distance to the nearest 0.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(update_matrix, [
    (([[0, 0, 0], [0, 1, 0], [0, 0, 0]],), [[0, 0, 0], [0, 1, 0], [0, 0, 0]]),
    (([[0, 0, 0], [0, 1, 0], [1, 1, 1]],), [[0, 0, 0], [0, 1, 0], [1, 2, 1]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

BFS from every 1 to its nearest 0 separately.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Start BFS from all zeros at once. The first time a cell is reached gives its shortest distance.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** BFS from every 1 to its nearest 0 separately.

```python
def update_matrix(mat):
    from collections import deque
    m, n = len(mat), len(mat[0])
    out = [[0] * n for _ in range(m)]
    for r in range(m):
        for c in range(n):
            if mat[r][c]:
                seen, q = {(r, c)}, deque([(r, c, 0)])
                while q:
                    x, y, d = q.popleft()
                    if mat[x][y] == 0:
                        out[r][c] = d
                        break
                    for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                        nx, ny = x + dx, y + dy
                        if 0 <= nx < m and 0 <= ny < n and (nx, ny) not in seen:
                            seen.add((nx, ny))
                            q.append((nx, ny, d + 1))
    return out
```

**Cost:** O((m·n)²) time, O(m·n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Start BFS from all zeros at once. The first time a cell is reached gives its shortest distance.

```python
def update_matrix(mat):
    from collections import deque
    m, n = len(mat), len(mat[0])
    dist = [[0 if mat[r][c] == 0 else -1 for c in range(n)] for r in range(m)]
    q = deque((r, c) for r in range(m) for c in range(n) if mat[r][c] == 0)
    while q:
        layer = []
        for _ in range(len(q)):
            r, c = q.popleft()
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and dist[nr][nc] == -1:
                    dist[nr][nc] = dist[r][c] + 1
                    q.append((nr, nc))
                    layer.append((nr, nc))
        if layer:
            pass
    return dist
```

**Cost:** O(m·n) time, O(m·n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: BFS from every 1 to its nearest 0 separately. That costs O((m·n)²) time and O(m·n) space."

"I can do better: Start BFS from all zeros at once. The first time a cell is reached gives its shortest distance. That's O(m·n) time and O(m·n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `mat = [[0, 1, 1, 1], [1, 1, 1, 1], [1, 1, 0, 1]]`

1. Cells at distance 1
2. Cells at distance 2
3. Cells at distance 3

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/graphs/01-matrix)

<a id="q6"></a>
### Q6. Surrounded Regions

🟡 Interview standard · *Search from the border*

Flip every 'O' region that is completely surrounded by 'X' (not touching the border) to 'X'. Modify the board in place.

**Examples**

- `board = [['X', 'X', 'X', 'X'], ['X', 'O', 'O', 'X'], ['X', 'X', 'O', 'X'], ['X', 'O', 'X', 'X']]` → `[['X', 'X', 'X', 'X'], ['X', 'X', 'X', 'X'], ['X', 'X', 'X', 'X'], ['X', 'O', 'X', 'X']]`
- `board = [['X']]` → `[['X']]`

In [ ]:
def solve_regions(board):
    """
    Flip every 'O' region that is completely surrounded by 'X' (not touching the
    border) to 'X'. Modify the board in place.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(solve_regions, [(([["X", "X", "X", "X"], ["X", "O", "O", "X"], ["X", "X", "O", "X"], ["X", "O", "X", "X"]],), [["X", "X", "X", "X"], ["X", "X", "X", "X"], ["X", "X", "X", "X"], ["X", "O", "X", "X"]]), (([["X"]],), [["X"]])], inplace=0)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For every 'O', BFS its region to check whether it reaches the border.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Flip the question: DFS from every border 'O' and mark it safe ('S'). Then every remaining 'O' is surrounded → 'X', and every 'S' goes back to 'O'.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For every 'O', BFS its region to check whether it reaches the border.

```python
def solve_regions(board):
    from collections import deque
    m, n = len(board), len(board[0])
    flip = []
    for r in range(m):
        for c in range(n):
            if board[r][c] == "O":
                seen, q, edge = {(r, c)}, deque([(r, c)]), False
                while q:
                    x, y = q.popleft()
                    if x in (0, m - 1) or y in (0, n - 1):
                        edge = True
                    for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                        nx, ny = x + dx, y + dy
                        if 0 <= nx < m and 0 <= ny < n and board[nx][ny] == "O" and (nx, ny) not in seen:
                            seen.add((nx, ny))
                            q.append((nx, ny))
                if not edge:
                    flip.append((r, c))
    for r, c in flip:
        board[r][c] = "X"
```

**Cost:** O((m·n)²) time, O(m·n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Flip the question: DFS from every border 'O' and mark it safe ('S'). Then every remaining 'O' is surrounded → 'X', and every 'S' goes back to 'O'.

```python
def solve_regions(board):
    m, n = len(board), len(board[0])

    def mark(r, c):
        if not (0 <= r < m and 0 <= c < n) or board[r][c] != "O":
            return
        board[r][c] = "S"
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            mark(r + dr, c + dc)

    for r in range(m):
        for c in range(n):
            if (r in (0, m - 1) or c in (0, n - 1)) and board[r][c] == "O":
                mark(r, c)
    for r in range(m):
        for c in range(n):
            board[r][c] = "O" if board[r][c] == "S" else "X"
```

**Cost:** O(m·n) time, O(m·n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For every 'O', BFS its region to check whether it reaches the border. That costs O((m·n)²) time and O(m·n) space."

"I can do better: Flip the question: DFS from every border 'O' and mark it safe ('S'). Then every remaining 'O' is surrounded → 'X', and every 'S' goes back to 'O'. That's O(m·n) time and O(m·n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/graphs/surrounded-regions)

<a id="q7"></a>
### Q7. Pacific Atlantic Water Flow

🟡 Interview standard · *Reverse DFS from both oceans*

Water flows to a neighbour of equal or lower height. Pacific touches the top and left edges, Atlantic the bottom and right. Return cells that can reach both oceans.

**Examples**

- `heights = [[1, 2, 2, 3, 5], [3, 2, 3, 4, 4], [2, 4, 5, 3, 1], [6, 7, 1, 4, 5], [5, 1, 1, 2, 4]]` → `[[0, 4], [1, 3], [1, 4], [2, 2], [3, 0], [3, 1], [4, 0]]`
- `heights = [[1]]` → `[[0, 0]]`

In [ ]:
def pacific_atlantic(heights):
    """
    Water flows to a neighbour of equal or lower height. Pacific touches the top
    and left edges, Atlantic the bottom and right. Return cells that can reach
    both oceans.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pacific_atlantic, [(([[1, 2, 2, 3, 5], [3, 2, 3, 4, 4], [2, 4, 5, 3, 1], [6, 7, 1, 4, 5], [5, 1, 1, 2, 4]],), [[0, 4], [1, 3], [1, 4], [2, 2], [3, 0], [3, 1], [4, 0]]), (([[1]],), [[0, 0]])], norm=sorted)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

From every cell, DFS downhill and record which oceans it reaches.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Search uphill from each ocean's edge cells, collecting everything that can drain into it. The answer is the intersection of the two sets.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** From every cell, DFS downhill and record which oceans it reaches.

```python
def pacific_atlantic(heights):
    m, n = len(heights), len(heights[0])
    res = []
    for r in range(m):
        for c in range(n):
            seen, stack, pac, atl = {(r, c)}, [(r, c)], False, False
            while stack:
                x, y = stack.pop()
                pac |= x == 0 or y == 0
                atl |= x == m - 1 or y == n - 1
                for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < m and 0 <= ny < n and (nx, ny) not in seen and heights[nx][ny] <= heights[x][y]:
                        seen.add((nx, ny))
                        stack.append((nx, ny))
            if pac and atl:
                res.append([r, c])
    return res
```

**Cost:** O((m·n)²) time, O(m·n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Search uphill from each ocean's edge cells, collecting everything that can drain into it. The answer is the intersection of the two sets.

```python
def pacific_atlantic(heights):
    m, n = len(heights), len(heights[0])

    def climb(starts):
        seen, stack = set(starts), list(starts)
        while stack:
            r, c = stack.pop()
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and (nr, nc) not in seen and heights[nr][nc] >= heights[r][c]:
                    seen.add((nr, nc))
                    stack.append((nr, nc))
        return seen

    pac = climb([(0, c) for c in range(n)] + [(r, 0) for r in range(m)])
    atl = climb([(m - 1, c) for c in range(n)] + [(r, n - 1) for r in range(m)])
    both = sorted(pac & atl)
    return [list(x) for x in both]
```

**Cost:** O(m·n) time, O(m·n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: From every cell, DFS downhill and record which oceans it reaches. That costs O((m·n)²) time and O(m·n) space."

"I can do better: Search uphill from each ocean's edge cells, collecting everything that can drain into it. The answer is the intersection of the two sets. That's O(m·n) time and O(m·n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `heights = [[1, 2, 2, 3, 5], [3, 2, 3, 4, 4], [2, 4, 5, 3, 1], [6, 7, 1, 4, 5], [5, 1, 1, 2, 4]]`

1. Cells that can drain into the Pacific (top / left)
2. Cells that can drain into the Atlantic (bottom / right)
3. Both oceans = intersection

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/graphs/pacific-atlantic-water-flow)

<a id="q8"></a>
### Q8. Number of Provinces

🟡 Interview standard · *Union–Find*

`isConnected[i][j] = 1` if cities i and j are directly connected. Count the connected groups (provinces).

**Examples**

- `is_connected = [[1, 1, 0], [1, 1, 0], [0, 0, 1]]` → `2`
- `is_connected = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]` → `3`

In [ ]:
def find_circle_num(is_connected):
    """
    isConnected[i][j] = 1 if cities i and j are directly connected. Count the
    connected groups (provinces).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_circle_num, [
    (([[1, 1, 0], [1, 1, 0], [0, 0, 1]],), 2),
    (([[1, 0, 0], [0, 1, 0], [0, 0, 1]],), 3),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute full reachability with Floyd–Warshall style closure, then count distinct reachability rows.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Union–find: union every connected pair; each successful union merges two provinces, so answer = n − unions. (DFS over the matrix is also O(n²).)

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Compute full reachability with Floyd–Warshall style closure, then count distinct reachability rows.

```python
def find_circle_num(is_connected):
    n = len(is_connected)
    reach = [[bool(is_connected[i][j]) or i == j for j in range(n)] for i in range(n)]
    for k in range(n):
        for i in range(n):
            for j in range(n):
                reach[i][j] = reach[i][j] or (reach[i][k] and reach[k][j])
    return len({tuple(row) for row in reach})
```

**Cost:** O(n³) time, O(n²) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Union–find: union every connected pair; each successful union merges two provinces, so answer = n − unions. (DFS over the matrix is also O(n²).)

```python
def find_circle_num(is_connected):
    n = len(is_connected)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    groups = n
    edges = [(i, j) for i in range(n) for j in range(i + 1, n) if is_connected[i][j]]
    for i, j in edges:
        a, b = find(i), find(j)
        if a != b:
            parent[a] = b
            groups -= 1
    return groups
```

**Cost:** O(n² · α(n)) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute full reachability with Floyd–Warshall style closure, then count distinct reachability rows. That costs O(n³) time and O(n²) space."

"I can do better: Union–find: union every connected pair; each successful union merges two provinces, so answer = n − unions. (DFS over the matrix is also O(n²).) That's O(n² · α(n)) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/graphs/number-of-provinces)

<a id="q9"></a>
### Q9. Is Graph Bipartite?

🟡 Interview standard · *BFS 2-colouring*

Can the nodes be split into two groups so every edge joins nodes from different groups? `graph[i]` lists i's neighbours.

**Examples**

- `graph = [[1, 2, 3], [0, 2], [0, 1, 3], [0, 2]]` → `False`
- `graph = [[1, 3], [0, 2], [1, 3], [0, 2]]` → `True`

In [ ]:
def is_bipartite(graph):
    """
    Can the nodes be split into two groups so every edge joins nodes from
    different groups? graph[i] lists i's neighbours.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_bipartite, [
    (([[1, 2, 3], [0, 2], [0, 1, 3], [0, 2]],), False),
    (([[1, 3], [0, 2], [1, 3], [0, 2]],), True),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try all 2ⁿ ways to assign the two colours.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Colour each uncoloured node 0, BFS, and give neighbours the opposite colour. A neighbour with the same colour proves it's not bipartite.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try all 2ⁿ ways to assign the two colours.

```python
def is_bipartite(graph):
    n = len(graph)
    for mask in range(1 << n):
        if all((mask >> u & 1) != (mask >> v & 1) for u in range(n) for v in graph[u]):
            return True
    return False
```

**Cost:** O(2ⁿ · E) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Colour each uncoloured node 0, BFS, and give neighbours the opposite colour. A neighbour with the same colour proves it's not bipartite.

```python
def is_bipartite(graph):
    from collections import deque
    n = len(graph)
    color = [-1] * n
    edges = [(u, v) for u in range(n) for v in graph[u] if u < v]
    for s in range(n):
        if color[s] != -1:
            continue
        color[s] = 0
        q = deque([s])
        while q:
            u = q.popleft()
            for v in graph[u]:
                if color[v] == -1:
                    color[v] = 1 - color[u]
                    q.append(v)
                elif color[v] == color[u]:
                    return False
    return True
```

**Cost:** O(V + E) time, O(V) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try all 2ⁿ ways to assign the two colours. That costs O(2ⁿ · E) time and O(n) space."

"I can do better: Colour each uncoloured node 0, BFS, and give neighbours the opposite colour. A neighbour with the same colour proves it's not bipartite. That's O(V + E) time and O(V) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `graph = [[1, 3], [0, 2], [1, 3], [0, 2]]`

1. Colour 0's neighbours with 1
2. Colour 1's neighbours with 0
3. Colour 3's neighbours with 0
4. Colour 2's neighbours with 1

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/graphs/is-graph-bipartite)

<a id="q10"></a>
### Q10. Course Schedule

🟡 Interview standard · *Kahn's topological sort*

There are `numCourses` courses; `[a, b]` means b must be taken before a. Can you finish all courses?

**Examples**

- `num_courses = 2, prerequisites = [[1, 0]]` → `True`
- `num_courses = 2, prerequisites = [[1, 0], [0, 1]]` → `False`
- `num_courses = 4, prerequisites = [[1, 0], [2, 1], [3, 2], [1, 3]]` → `False`

In [ ]:
def can_finish(num_courses, prerequisites):
    """
    There are numCourses courses; [a, b] means b must be taken before a. Can you
    finish all courses?
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(can_finish, [
    ((2, [[1, 0]]), True),
    ((2, [[1, 0], [0, 1]]), False),
    ((4, [[1, 0], [2, 1], [3, 2], [1, 3]]), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

From every course, DFS to look for a path back to itself (a cycle), with no memo.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Count in-degrees. Queue every course with in-degree 0, take it, and decrement its dependents. If every course gets taken there is no cycle.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** From every course, DFS to look for a path back to itself (a cycle), with no memo.

```python
def can_finish(num_courses, prerequisites):
    adj = {i: [] for i in range(num_courses)}
    for a, b in prerequisites:
        adj[b].append(a)

    def reaches(u, target, seen):
        for v in adj[u]:
            if v == target or (v not in seen and (seen.add(v) or reaches(v, target, seen))):
                return True
        return False
    return not any(reaches(c, c, set()) for c in range(num_courses))
```

**Cost:** O(V · (V + E)) time, O(V + E) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Count in-degrees. Queue every course with in-degree 0, take it, and decrement its dependents. If every course gets taken there is no cycle.

```python
def can_finish(num_courses, prerequisites):
    from collections import deque
    adj = [[] for _ in range(num_courses)]
    indeg = [0] * num_courses
    for a, b in prerequisites:
        adj[b].append(a)
        indeg[a] += 1
    q = deque(i for i in range(num_courses) if indeg[i] == 0)
    taken = []
    while q:
        u = q.popleft()
        taken.append(u)
        for v in adj[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return len(taken) == num_courses
```

**Cost:** O(V + E) time, O(V + E) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: From every course, DFS to look for a path back to itself (a cycle), with no memo. That costs O(V · (V + E)) time and O(V + E) space."

"I can do better: Count in-degrees. Queue every course with in-degree 0, take it, and decrement its dependents. If every course gets taken there is no cycle. That's O(V + E) time and O(V + E) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `num_courses = 5, prerequisites = [[1, 0], [2, 0], [3, 1], [3, 2], [4, 3]]`

1. Take course 0; its dependents lose one prerequisite
2. Take course 1; its dependents lose one prerequisite
3. Take course 2; its dependents lose one prerequisite
4. Take course 3; its dependents lose one prerequisite
5. Take course 4; its dependents lose one prerequisite

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/graphs/course-schedule)

<a id="q11"></a>
### Q11. Course Schedule II

🟡 Interview standard · *Topological order*

Return an order to take all courses respecting prerequisites, or `[]` if impossible.

**Examples**

- `num_courses = 2, prerequisites = [[1, 0]]` → `[0, 1]`
- `num_courses = 4, prerequisites = [[1, 0], [2, 0], [3, 1], [3, 2]]` → `[0, 2, 1, 3]`
- `num_courses = 2, prerequisites = [[0, 1], [1, 0]]` → `[]`

In [ ]:
def _topo_ok(n, pre, out):
    if not out:
        return _has_cycle(n, pre)
    pos = {c: i for i, c in enumerate(out)}
    return len(out) == n and len(pos) == n and all(pos[b] < pos[a] for a, b in pre)

def _has_cycle(n, pre):
    indeg = [0] * n
    adj = [[] for _ in range(n)]
    for a, b in pre:
        adj[b].append(a)
        indeg[a] += 1
    q = [i for i in range(n) if indeg[i] == 0]
    seen = 0
    while q:
        u = q.pop()
        seen += 1
        for v in adj[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return seen < n

def find_order(num_courses, prerequisites):
    """
    Return an order to take all courses respecting prerequisites, or [] if
    impossible.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_order, [((2, [[1, 0]]), [0, 1]), ((4, [[1, 0], [2, 0], [3, 1], [3, 2]]), [0, 2, 1, 3]), ((2, [[0, 1], [1, 0]]), [])], verify=lambda a, out: _topo_ok(a[0], a[1], out))

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Repeatedly scan for any untaken course whose prerequisites are all taken.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Kahn's algorithm: the order in which courses leave the zero-in-degree queue is a valid order.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Repeatedly scan for any untaken course whose prerequisites are all taken.

```python
def find_order(num_courses, prerequisites):
    taken, order = set(), []
    while len(order) < num_courses:
        for c in range(num_courses):
            if c not in taken and all(b in taken for a, b in prerequisites if a == c):
                taken.add(c)
                order.append(c)
                break
        else:
            return []
    return order
```

**Cost:** O(V · E) time, O(V) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Kahn's algorithm: the order in which courses leave the zero-in-degree queue is a valid order.

```python
def find_order(num_courses, prerequisites):
    from collections import deque
    adj = [[] for _ in range(num_courses)]
    indeg = [0] * num_courses
    for a, b in prerequisites:
        adj[b].append(a)
        indeg[a] += 1
    q = deque(i for i in range(num_courses) if indeg[i] == 0)
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in adj[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return order if len(order) == num_courses else []
```

**Cost:** O(V + E) time, O(V + E) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Repeatedly scan for any untaken course whose prerequisites are all taken. That costs O(V · E) time and O(V) space."

"I can do better: Kahn's algorithm: the order in which courses leave the zero-in-degree queue is a valid order. That's O(V + E) time and O(V + E) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `num_courses = 4, prerequisites = [[1, 0], [2, 0], [3, 1], [3, 2]]`

1. Course 0 is ready → add to the order
2. Course 1 is ready → add to the order
3. Course 2 is ready → add to the order
4. Course 3 is ready → add to the order

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/graphs/course-schedule-ii)

<a id="q12"></a>
### Q12. Find Eventual Safe States

🟡 Interview standard · *3-colour DFS*

A node is safe if every path from it ends at a terminal node (no outgoing edges). Return all safe nodes, sorted.

**Examples**

- `graph = [[1, 2], [2, 3], [5], [0], [5], [], []]` → `[2, 4, 5, 6]`
- `graph = [[1, 2, 3, 4], [1, 2], [3, 4], [0, 4], []]` → `[4]`

In [ ]:
def eventual_safe_nodes(graph):
    """
    A node is safe if every path from it ends at a terminal node (no outgoing
    edges). Return all safe nodes, sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(eventual_safe_nodes, [
    (([[1, 2], [2, 3], [5], [0], [5], [], []],), [2, 4, 5, 6]),
    (([[1, 2, 3, 4], [1, 2], [3, 4], [0, 4], []],), [4]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each node, explore every path from it and check none loops (no memo).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

DFS with three states: 0 unvisited, 1 on the current path, 2 known safe. Reaching a node in state 1 means a cycle, so everything on that path is unsafe.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For each node, explore every path from it and check none loops (no memo).

```python
def eventual_safe_nodes(graph):
    def safe(u, path):
        if u in path:
            return False
        return all(safe(v, path | {u}) for v in graph[u])
    return [u for u in range(len(graph)) if safe(u, frozenset())]
```

**Cost:** O(V · (V + E)) time, O(V) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** DFS with three states: 0 unvisited, 1 on the current path, 2 known safe. Reaching a node in state 1 means a cycle, so everything on that path is unsafe.

```python
def eventual_safe_nodes(graph):
    n = len(graph)
    state = [0] * n
    edges = [(u, v) for u in range(n) for v in graph[u]]

    def dfs(u):
        if state[u]:
            return state[u] == 2
        state[u] = 1
        for v in graph[u]:
            if not dfs(v):
                return False
        state[u] = 2
        return True

    return [u for u in range(n) if dfs(u)]
```

**Cost:** O(V + E) time, O(V) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each node, explore every path from it and check none loops (no memo). That costs O(V · (V + E)) time and O(V) space."

"I can do better: DFS with three states: 0 unvisited, 1 on the current path, 2 known safe. Reaching a node in state 1 means a cycle, so everything on that path is unsafe. That's O(V + E) time and O(V) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `graph = [[1, 2], [2, 3], [5], [0], [5], [], []]`

1. Every path from 5 ends → safe
2. Every path from 2 ends → safe
3. 3 can reach a cycle → unsafe
4. 1 can reach a cycle → unsafe
5. 0 can reach a cycle → unsafe
6. Every path from 4 ends → safe
7. Every path from 6 ends → safe

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/graphs/find-eventual-safe-states)

<a id="q13"></a>
### Q13. Redundant Connection

🟡 Interview standard · *Union–Find detects the cycle edge*

A tree of n nodes had one extra edge added. Return the edge that can be removed to make it a tree again (the last such edge in the input).

**Examples**

- `edges = [[1, 2], [1, 3], [2, 3]]` → `[2, 3]`
- `edges = [[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]]` → `[1, 4]`

In [ ]:
def find_redundant_connection(edges):
    """
    A tree of n nodes had one extra edge added. Return the edge that can be
    removed to make it a tree again (the last such edge in the input).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_redundant_connection, [
    (([[1, 2], [1, 3], [2, 3]],), [2, 3]),
    (([[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]],), [1, 4]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try removing each edge (from the last) and check whether the rest is connected.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Add edges one by one with union–find. The first edge whose endpoints already share a root closes a cycle — that's the answer.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try removing each edge (from the last) and check whether the rest is connected.

```python
def find_redundant_connection(edges):
    n = len(edges)
    for skip in range(n - 1, -1, -1):
        adj = {i: [] for i in range(1, n + 1)}
        for k, (a, b) in enumerate(edges):
            if k != skip:
                adj[a].append(b)
                adj[b].append(a)
        seen, stack = {1}, [1]
        while stack:
            for v in adj[stack.pop()]:
                if v not in seen:
                    seen.add(v)
                    stack.append(v)
        if len(seen) == n:
            return edges[skip]
```

**Cost:** O(E · (V + E)) time, O(V + E) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Add edges one by one with union–find. The first edge whose endpoints already share a root closes a cycle — that's the answer.

```python
def find_redundant_connection(edges):
    parent = list(range(len(edges) + 1))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for k, (a, b) in enumerate(edges):
        ra, rb = find(a), find(b)
        if ra == rb:
            return [a, b]
        parent[ra] = rb
```

**Cost:** O(E · α(V)) time, O(V) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try removing each edge (from the last) and check whether the rest is connected. That costs O(E · (V + E)) time and O(V + E) space."

"I can do better: Add edges one by one with union–find. The first edge whose endpoints already share a root closes a cycle — that's the answer. That's O(E · α(V)) time and O(V) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `edges = [[1, 2], [1, 3], [2, 3]]`

1. Union 1 and 2
2. Union 1 and 3
3. 2 and 3 are already connected → [2, 3] is redundant

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/graphs/redundant-connection)

<a id="q14"></a>
### Q14. Accounts Merge

🟡 Interview standard · *Union–Find over emails*

Each account is [name, emails…]. Accounts sharing any email belong to the same person. Merge them; return [name, sorted emails…] per person.

**Examples**

- `accounts = [['John', 'a@m.co', 'b@m.co'], ['John', 'a@m.co', 'c@m.co'], ['Mary', 'm@m.co'], ['John', 'j@m.co']]` → `[['John', 'a@m.co', 'b@m.co', 'c@m.co'], ['Mary', 'm@m.co'], ['John', 'j@m.co']]`

In [ ]:
def accounts_merge(accounts):
    """
    Each account is [name, emails…]. Accounts sharing any email belong to the
    same person. Merge them; return [name, sorted emails…] per person.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(accounts_merge, [(([["John", "a@m.co", "b@m.co"], ["John", "a@m.co", "c@m.co"], ["Mary", "m@m.co"], ["John", "j@m.co"]],), [["John", "a@m.co", "b@m.co", "c@m.co"], ["Mary", "m@m.co"], ["John", "j@m.co"]])], norm=sorted)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Repeatedly find two accounts that share an email and merge them, until no merge happens.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Union each account's emails with its first email. Then group emails by their root, sort, and attach the owner's name.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Repeatedly find two accounts that share an email and merge them, until no merge happens.

```python
def accounts_merge(accounts):
    groups = [[a[0], set(a[1:])] for a in accounts]
    merged = True
    while merged:
        merged = False
        for i in range(len(groups)):
            for j in range(i + 1, len(groups)):
                if groups[i][1] & groups[j][1]:
                    groups[i][1] |= groups.pop(j)[1]
                    merged = True
                    break
            if merged:
                break
    return [[name] + sorted(emails) for name, emails in groups]
```

**Cost:** O(n² · k) per pass time, O(n · k) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Union each account's emails with its first email. Then group emails by their root, sort, and attach the owner's name.

```python
def accounts_merge(accounts):
    parent, owner = {}, {}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for name, *emails in accounts:
        for e in emails:
            parent.setdefault(e, e)
            owner[e] = name
        for e in emails[1:]:
            parent[find(e)] = find(emails[0])
    groups = {}
    for e in parent:
        groups.setdefault(find(e), []).append(e)
    return [[owner[r]] + sorted(g) for r, g in groups.items()]
```

**Cost:** O(N log N), N = total emails time, O(N) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Repeatedly find two accounts that share an email and merge them, until no merge happens. That costs O(n² · k) per pass time and O(n · k) space."

"I can do better: Union each account's emails with its first email. Then group emails by their root, sort, and attach the owner's name. That's O(N log N), N = total emails time and O(N) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `accounts = [['John', 'a@m.co', 'b@m.co'], ['John', 'a@m.co', 'c@m.co'], ['Mary', 'm@m.co'], ['John', 'j@m.co']]`

1. Union John's emails with a@m.co
2. Union John's emails with a@m.co
3. Union Mary's emails with m@m.co
4. Union John's emails with j@m.co
5. Group emails by their root

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/graphs/accounts-merge)

<a id="q15"></a>
### Q15. Shortest Path in Binary Matrix

🟡 Interview standard · *BFS, 8 directions*

Return the length (cells visited) of the shortest clear path of 0s from top-left to bottom-right, moving in 8 directions, or −1.

**Examples**

- `grid = [[0, 1], [1, 0]]` → `2`
- `grid = [[0, 0, 0], [1, 1, 0], [1, 1, 0]]` → `4`
- `grid = [[1, 0, 0], [1, 1, 0], [1, 1, 0]]` → `-1`

In [ ]:
def shortest_path_binary_matrix(grid):
    """
    Return the length (cells visited) of the shortest clear path of 0s from top-
    left to bottom-right, moving in 8 directions, or −1.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(shortest_path_binary_matrix, [
    (([[0, 1], [1, 0]],), 2),
    (([[0, 0, 0], [1, 1, 0], [1, 1, 0]],), 4),
    (([[1, 0, 0], [1, 1, 0], [1, 1, 0]],), -1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

DFS every simple path and keep the shortest.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

BFS from (0, 0) over the 8 neighbours. The first time BFS reaches the corner, that layer number is the shortest length.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** DFS every simple path and keep the shortest.

```python
def shortest_path_binary_matrix(grid):
    n = len(grid)
    if grid[0][0] or grid[n - 1][n - 1]:
        return -1
    best = [float("inf")]

    def dfs(r, c, length, seen):
        if length >= best[0]:
            return
        if (r, c) == (n - 1, n - 1):
            best[0] = length
            return
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                nr, nc = r + dr, c + dc
                if 0 <= nr < n and 0 <= nc < n and not grid[nr][nc] and (nr, nc) not in seen:
                    seen.add((nr, nc))
                    dfs(nr, nc, length + 1, seen)
                    seen.remove((nr, nc))
    dfs(0, 0, 1, {(0, 0)})
    return -1 if best[0] == float("inf") else best[0]
```

**Cost:** O(8^(m·n)) time, O(m·n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** BFS from (0, 0) over the 8 neighbours. The first time BFS reaches the corner, that layer number is the shortest length.

```python
def shortest_path_binary_matrix(grid):
    from collections import deque
    n = len(grid)
    if grid[0][0] or grid[n - 1][n - 1]:
        return -1
    q, seen, length = deque([(0, 0)]), {(0, 0)}, 1
    while q:
        for _ in range(len(q)):
            r, c = q.popleft()
            if (r, c) == (n - 1, n - 1):
                return length
            for dr in (-1, 0, 1):
                for dc in (-1, 0, 1):
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < n and 0 <= nc < n and not grid[nr][nc] and (nr, nc) not in seen:
                        seen.add((nr, nc))
                        q.append((nr, nc))
        length += 1
    return -1
```

**Cost:** O(m·n) time, O(m·n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: DFS every simple path and keep the shortest. That costs O(8^(m·n)) time and O(m·n) space."

"I can do better: BFS from (0, 0) over the 8 neighbours. The first time BFS reaches the corner, that layer number is the shortest length. That's O(m·n) time and O(m·n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `grid = [[0, 0, 0, 1], [1, 1, 0, 1], [0, 0, 0, 0], [0, 1, 1, 0]]`

1. Layer 1
2. Layer 2
3. Layer 3
4. Layer 4
5. Layer 5

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/graphs/shortest-path-in-binary-matrix)

<a id="q16"></a>
### Q16. Network Delay Time

🟡 Interview standard · *Dijkstra*

`times[i] = (u, v, w)` is a directed edge taking w. A signal starts at node `k`. Return the time until all n nodes have it, or −1.

**Examples**

- `times = [[2, 1, 1], [2, 3, 1], [3, 4, 1]], n = 4, k = 2` → `2`
- `times = [[1, 2, 1]], n = 2, k = 1` → `1`
- `times = [[1, 2, 1]], n = 2, k = 2` → `-1`

In [ ]:
def network_delay_time(times, n, k):
    """
    times[i] = (u, v, w) is a directed edge taking w. A signal starts at node k.
    Return the time until all n nodes have it, or −1.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(network_delay_time, [
    (([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4, 2), 2),
    (([[1, 2, 1]], 2, 1), 1),
    (([[1, 2, 1]], 2, 2), -1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Bellman–Ford: relax every edge n − 1 times.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Dijkstra: pop the closest unfinished node from a min-heap and relax its edges. Skip stale heap entries. The answer is the largest final distance.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Bellman–Ford: relax every edge n − 1 times.

```python
def network_delay_time(times, n, k):
    dist = [float("inf")] * (n + 1)
    dist[k] = 0
    for _ in range(n - 1):
        for u, v, w in times:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
    best = max(dist[1:])
    return -1 if best == float("inf") else best
```

**Cost:** O(V · E) time, O(V) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Dijkstra: pop the closest unfinished node from a min-heap and relax its edges. Skip stale heap entries. The answer is the largest final distance.

```python
def network_delay_time(times, n, k):
    import heapq
    adj = {i: [] for i in range(1, n + 1)}
    for u, v, w in times:
        adj[u].append((v, w))
    dist = {}
    heap = [(0, k)]
    while heap:
        d, u = heapq.heappop(heap)
        if u in dist:
            continue
        dist[u] = d
        for v, w in adj[u]:
            if v not in dist:
                heapq.heappush(heap, (d + w, v))
    return max(dist.values()) if len(dist) == n else -1
```

**Cost:** O(E log V) time, O(V + E) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Bellman–Ford: relax every edge n − 1 times. That costs O(V · E) time and O(V) space."

"I can do better: Dijkstra: pop the closest unfinished node from a min-heap and relax its edges. Skip stale heap entries. The answer is the largest final distance. That's O(E log V) time and O(V + E) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `times = [[1, 2, 4], [1, 3, 1], [3, 2, 2], [2, 4, 1], [3, 4, 5]], n = 4, k = 1`

1. Finalise node 1 at time 0
2. Finalise node 3 at time 1
3. Finalise node 2 at time 3
4. Finalise node 4 at time 4

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/graphs/network-delay-time)

<a id="q17"></a>
### Q17. Cheapest Flights Within K Stops

🟡 Interview standard · *Bellman–Ford, k + 1 rounds*

Find the cheapest price from `src` to `dst` using at most `k` stops, or −1.

**Examples**

- `n = 4, flights = [[0, 1, 100], [1, 2, 100], [2, 0, 100], [1, 3, 600], [2, 3, 200]], src = 0, dst = 3, k = 1` → `700`
- `n = 3, flights = [[0, 1, 100], [1, 2, 100], [0, 2, 500]], src = 0, dst = 2, k = 1` → `200`
- `n = 3, flights = [[0, 1, 100], [1, 2, 100], [0, 2, 500]], src = 0, dst = 2, k = 0` → `500`

In [ ]:
def find_cheapest_price(n, flights, src, dst, k):
    """
    Find the cheapest price from src to dst using at most k stops, or −1.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_cheapest_price, [
    ((4, [[0, 1, 100], [1, 2, 100], [2, 0, 100], [1, 3, 600], [2, 3, 200]], 0, 3, 1), 700),
    ((3, [[0, 1, 100], [1, 2, 100], [0, 2, 500]], 0, 2, 1), 200),
    ((3, [[0, 1, 100], [1, 2, 100], [0, 2, 500]], 0, 2, 0), 500),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

DFS every route of at most k + 1 flights and keep the cheapest.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Run k + 1 rounds of relaxing every flight, always reading prices from the previous round's copy so each round adds at most one flight.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** DFS every route of at most k + 1 flights and keep the cheapest.

```python
def find_cheapest_price(n, flights, src, dst, k):
    adj = {i: [] for i in range(n)}
    for u, v, w in flights:
        adj[u].append((v, w))
    best = [float("inf")]

    def dfs(u, cost, used):
        if u == dst:
            best[0] = min(best[0], cost)
            return
        if used == k + 1:
            return
        for v, w in adj[u]:
            dfs(v, cost + w, used + 1)
    dfs(src, 0, 0)
    return -1 if best[0] == float("inf") else best[0]
```

**Cost:** O(n^(k+1)) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Run k + 1 rounds of relaxing every flight, always reading prices from the previous round's copy so each round adds at most one flight.

```python
def find_cheapest_price(n, flights, src, dst, k):
    INF = float("inf")
    price = [INF] * n
    price[src] = 0
    for rnd in range(k + 1):
        nxt = price[:]
        for u, v, w in flights:
            if price[u] + w < nxt[v]:
                nxt[v] = price[u] + w
        price = nxt
    return -1 if price[dst] == INF else price[dst]
```

**Cost:** O(k · E) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: DFS every route of at most k + 1 flights and keep the cheapest. That costs O(n^(k+1)) time and O(k) space."

"I can do better: Run k + 1 rounds of relaxing every flight, always reading prices from the previous round's copy so each round adds at most one flight. That's O(k · E) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `n = 5, flights = [[0, 1, 100], [1, 2, 100], [2, 4, 100], [0, 3, 500], [3, 4, 50], [1, 3, 100]], src = 0, dst = 4, k = 2`

1. After round 1 (≤ 1 flights)
2. After round 2 (≤ 2 flights)
3. After round 3 (≤ 3 flights)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/graphs/cheapest-flights-within-k-stops)

<a id="q18"></a>
### Q18. Path With Minimum Effort

🟡 Interview standard · *Dijkstra on max edge*

A route's effort is its largest absolute height difference between consecutive cells. Return the minimum effort from top-left to bottom-right.

**Examples**

- `heights = [[1, 2, 2], [3, 8, 2], [5, 3, 5]]` → `2`
- `heights = [[1, 2, 3], [3, 8, 4], [5, 3, 5]]` → `1`
- `heights = [[1, 2, 1, 1, 1], [1, 2, 1, 2, 1], [1, 2, 1, 2, 1], [1, 2, 1, 2, 1], [1, 1, 1, 2, 1]]` → `0`

In [ ]:
def minimum_effort_path(heights):
    """
    A route's effort is its largest absolute height difference between
    consecutive cells. Return the minimum effort from top-left to bottom-right.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(minimum_effort_path, [
    (([[1, 2, 2], [3, 8, 2], [5, 3, 5]],), 2),
    (([[1, 2, 3], [3, 8, 4], [5, 3, 5]],), 1),
    (([[1, 2, 1, 1, 1], [1, 2, 1, 2, 1], [1, 2, 1, 2, 1], [1, 2, 1, 2, 1], [1, 1, 1, 2, 1]],), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Relax all cells repeatedly (Bellman–Ford style) until no effort improves.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Dijkstra where a path's cost is the max edge so far: pop the cell with the smallest effort, relax neighbours with max(effort, |Δh|).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Relax all cells repeatedly (Bellman–Ford style) until no effort improves.

```python
def minimum_effort_path(heights):
    m, n = len(heights), len(heights[0])
    eff = [[float("inf")] * n for _ in range(m)]
    eff[0][0] = 0
    changed = True
    while changed:
        changed = False
        for r in range(m):
            for c in range(n):
                for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < m and 0 <= nc < n:
                        e = max(eff[r][c], abs(heights[nr][nc] - heights[r][c]))
                        if e < eff[nr][nc]:
                            eff[nr][nc] = e
                            changed = True
    return eff[m - 1][n - 1]
```

**Cost:** O((m·n)²) time, O(m·n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Dijkstra where a path's cost is the max edge so far: pop the cell with the smallest effort, relax neighbours with max(effort, |Δh|).

```python
def minimum_effort_path(heights):
    import heapq
    m, n = len(heights), len(heights[0])
    eff = [[float("inf")] * n for _ in range(m)]
    eff[0][0] = 0
    heap, done = [(0, 0, 0)], []
    while heap:
        e, r, c = heapq.heappop(heap)
        if e > eff[r][c]:
            continue
        done.append((r, c))
        if (r, c) == (m - 1, n - 1):
            return e
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n:
                ne = max(e, abs(heights[nr][nc] - heights[r][c]))
                if ne < eff[nr][nc]:
                    eff[nr][nc] = ne
                    heapq.heappush(heap, (ne, nr, nc))
```

**Cost:** O(m·n · log(m·n)) time, O(m·n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Relax all cells repeatedly (Bellman–Ford style) until no effort improves. That costs O((m·n)²) time and O(m·n) space."

"I can do better: Dijkstra where a path's cost is the max edge so far: pop the cell with the smallest effort, relax neighbours with max(effort, |Δh|). That's O(m·n · log(m·n)) time and O(m·n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `heights = [[1, 2, 2], [3, 8, 2], [5, 3, 5]]`

1. Settle (0,0) with effort 0
2. Settle (0,1) with effort 1
3. Settle (0,2) with effort 1
4. Settle (1,2) with effort 1
5. Settle (1,0) with effort 2
6. Settle (2,0) with effort 2
7. Settle (2,1) with effort 2
8. Settle (2,2) with effort 2

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/graphs/path-with-minimum-effort)

<a id="q19"></a>
### Q19. Min Cost to Connect All Points

🟡 Interview standard · *Prim's MST (dense)*

Connect all points with edges costing their Manhattan distance; return the minimum total cost.

**Examples**

- `points = [[0, 0], [2, 2], [3, 10], [5, 2], [7, 0]]` → `20`
- `points = [[3, 12], [-2, 5], [-4, 1]]` → `18`

In [ ]:
def min_cost_connect_points(points):
    """
    Connect all points with edges costing their Manhattan distance; return the
    minimum total cost.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(min_cost_connect_points, [
    (([[0, 0], [2, 2], [3, 10], [5, 2], [7, 0]],), 20),
    (([[3, 12], [-2, 5], [-4, 1]],), 18),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Kruskal: build all n² edges, sort them, add the cheapest that joins two components.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Prim for a dense graph: keep each point's cheapest connection to the tree in an array; repeatedly add the closest point and update the others. No heap or edge list needed.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Kruskal: build all n² edges, sort them, add the cheapest that joins two components.

```python
def min_cost_connect_points(points):
    n = len(points)
    edges = sorted((abs(points[i][0] - points[j][0]) + abs(points[i][1] - points[j][1]), i, j) for i in range(n) for j in range(i + 1, n))
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x
    total = 0
    for w, i, j in edges:
        a, b = find(i), find(j)
        if a != b:
            parent[a] = b
            total += w
    return total
```

**Cost:** O(n² log n) time, O(n²) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Prim for a dense graph: keep each point's cheapest connection to the tree in an array; repeatedly add the closest point and update the others. No heap or edge list needed.

```python
def min_cost_connect_points(points):
    n = len(points)
    INF = float("inf")
    best = [INF] * n
    best[0] = 0
    in_tree = [False] * n
    tree_edges, link = [], [-1] * n
    total = 0
    for _ in range(n):
        u = min((i for i in range(n) if not in_tree[i]), key=lambda i: best[i])
        in_tree[u] = True
        total += best[u]
        if link[u] >= 0:
            tree_edges.append((link[u], u, best[u]))
        for v in range(n):
            d = abs(points[u][0] - points[v][0]) + abs(points[u][1] - points[v][1])
            if not in_tree[v] and d < best[v]:
                best[v], link[v] = d, u
    return total
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Kruskal: build all n² edges, sort them, add the cheapest that joins two components. That costs O(n² log n) time and O(n²) space."

"I can do better: Prim for a dense graph: keep each point's cheapest connection to the tree in an array; repeatedly add the closest point and update the others. No heap or edge list needed. That's O(n²) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `points = [[0, 0], [2, 2], [3, 10], [5, 2], [7, 0]]`

1. Add point 0 (cost 0); update everyone's cheapest link
2. Add point 1 (cost 4); update everyone's cheapest link
3. Add point 3 (cost 3); update everyone's cheapest link
4. Add point 4 (cost 4); update everyone's cheapest link
5. Add point 2 (cost 9); update everyone's cheapest link

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/graphs/min-cost-to-connect-all-points)

<a id="q20"></a>
### Q20. Word Ladder

🔴 Stretch · *BFS with wildcard buckets*

Transform `beginWord` to `endWord` changing one letter at a time, every step a word in `wordList`. Return the number of words in the shortest sequence, or 0.

**Examples**

- `begin_word = 'hit', end_word = 'cog', word_list = ['hot', 'dot', 'dog', 'lot', 'log', 'cog']` → `5`
- `begin_word = 'hit', end_word = 'cog', word_list = ['hot', 'dot', 'dog', 'lot', 'log']` → `0`

In [ ]:
def ladder_length(begin_word, end_word, word_list):
    """
    Transform beginWord to endWord changing one letter at a time, every step a
    word in wordList. Return the number of words in the shortest sequence, or 0.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(ladder_length, [
    (('hit', 'cog', ['hot', 'dot', 'dog', 'lot', 'log', 'cog']), 5),
    (('hit', 'cog', ['hot', 'dot', 'dog', 'lot', 'log']), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

BFS where each word's neighbours are found by comparing it with every word in the list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

BFS, but find neighbours through patterns: 'hot' belongs to buckets '\*ot', 'h\*t', 'ho\*'. Words sharing a bucket differ by one letter.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** BFS where each word's neighbours are found by comparing it with every word in the list.

```python
def ladder_length(begin_word, end_word, word_list):
    from collections import deque
    words = set(word_list)
    if end_word not in words:
        return 0
    q, seen = deque([(begin_word, 1)]), {begin_word}
    while q:
        w, d = q.popleft()
        if w == end_word:
            return d
        for x in words:
            if x not in seen and sum(a != b for a, b in zip(w, x)) == 1:
                seen.add(x)
                q.append((x, d + 1))
    return 0
```

**Cost:** O(N² · L) time, O(N) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** BFS, but find neighbours through patterns: 'hot' belongs to buckets '\*ot', 'h\*t', 'ho\*'. Words sharing a bucket differ by one letter.

```python
def ladder_length(begin_word, end_word, word_list):
    from collections import deque, defaultdict
    if end_word not in word_list:
        return 0
    L = len(begin_word)
    buckets = defaultdict(list)
    for w in word_list:
        for i in range(L):
            buckets[w[:i] + "*" + w[i + 1:]].append(w)
    q, seen, steps = deque([begin_word]), {begin_word}, 1
    while q:
        for _ in range(len(q)):
            w = q.popleft()
            if w == end_word:
                return steps
            for i in range(L):
                for nxt in buckets[w[:i] + "*" + w[i + 1:]]:
                    if nxt not in seen:
                        seen.add(nxt)
                        q.append(nxt)
        steps += 1
    return 0
```

**Cost:** O(N · L²) time, O(N · L²) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: BFS where each word's neighbours are found by comparing it with every word in the list. That costs O(N² · L) time and O(N) space."

"I can do better: BFS, but find neighbours through patterns: 'hot' belongs to buckets '\*ot', 'h\*t', 'ho\*'. Words sharing a bucket differ by one letter. That's O(N · L²) time and O(N · L²) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `begin_word = 'hit', end_word = 'cog', word_list = ['hot', 'dot', 'dog', 'lot', 'log', 'cog']`

1. Step 1: frontier ['hit']
2. Step 2: frontier ['hot']
3. Step 3: frontier ['dot', 'lot']
4. Step 4: frontier ['dog', 'log']
5. Step 5: frontier ['cog']

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/graphs/word-ladder)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. BFS vs DFS — when do you use each?</b></summary>

BFS when you need the shortest path or minimum steps in an unweighted graph, or level-by-level processing. DFS for exploring all reachable nodes, connected components, cycle detection, topological sort and backtracking-style searches. Both are O(V + E).

</details>

<details>
<summary><b>2. Adjacency list or adjacency matrix?</b></summary>

List for sparse graphs (most real graphs): O(V + E) space and fast neighbour iteration. Matrix for dense graphs or when you need O(1) edge checks, at O(V²) space.

</details>

<details>
<summary><b>3. Why does Dijkstra not work with negative weights?</b></summary>

Dijkstra permanently fixes a vertex's distance when it's popped as the closest. A negative edge found later could reduce that distance, but the algorithm never revisits it. Use Bellman–Ford instead.

</details>

<details>
<summary><b>4. How do you detect a cycle in a directed graph?</b></summary>

DFS with three colours (unvisited, on stack, done): an edge to an 'on stack' vertex is a back edge → cycle. Or Kahn's algorithm: if the topological order doesn't include all vertices, there is a cycle.

</details>

<details>
<summary><b>5. What is a topological sort and when does it exist?</b></summary>

A linear order of vertices such that every directed edge u → v has u before v. It exists if and only if the graph is a DAG (no directed cycles).

</details>

<details>
<summary><b>6. Kruskal vs Prim?</b></summary>

Both build a minimum spanning tree. Kruskal sorts all edges and adds the cheapest that doesn't form a cycle (union-find) — good for sparse graphs. Prim grows one tree from a start vertex, adding the cheapest edge leaving it — good for dense graphs.

</details>

<details>
<summary><b>7. What is union-find and what are its optimisations?</b></summary>

A structure that tracks which set each element belongs to, supporting `find` (root of a set) and `union` (merge sets). Path compression and union by rank/size make both operations nearly O(1) (inverse Ackermann).

</details>

<details>
<summary><b>8. How many edges can a graph have?</b></summary>

Undirected simple graph: at most V(V − 1)/2. Directed: at most V(V − 1). A tree has exactly V − 1; a connected graph needs at least V − 1.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: What a graph is and how to store it
- [ ] I can explain: Depth-first search
- [ ] I can explain: Breadth-first search and shortest paths
- [ ] I can explain: Grids: islands and flood fill
- [ ] I can explain: Directed graphs: topological order and cycles
- [ ] I can explain: Union–Find (disjoint set union)
- [ ] I can explain: Weighted shortest paths
- [ ] I can explain: Spanning trees and two-colouring

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Network Delay Time
- [ ] [Q17](#q17) Cheapest Flights Within K Stops
- [ ] [Q18](#q18) Path With Minimum Effort
- [ ] [Q19](#q19) Min Cost to Connect All Points
- [ ] [Q20](#q20) Word Ladder

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.